# ShowStrain

Cepstral strain mapping of a 4D-STEM scan with `ShowStrain`. Each diffraction pattern is
intensity-scaled and Fourier transformed. The crystal periodicity appears as a lattice of
peaks in the transform, and their positions at every scan position give the local
lattice and the strain.

The widget drives `quantem.diffraction.StrainMapAutocorrelation`. Every control is one of
its keyword arguments, so the analysis can always be repeated in code.

This tutorial uses a small synthetic scan: Gaussian Bragg disks on a square lattice whose
real-space spacing along the scan columns grows by 1 % from left to right.

In [ ]:
import numpy as np

from quantem.core.datastructures import Dataset4dstem
from quantem.widget import ShowStrain


def synthetic_lattice_4dstem(scan_shape=(24, 16), det=96, spacing=12.0, max_strain=0.01, seed=0):
    """Bragg disks on a square lattice; the real-space column spacing grows by max_strain across the scan."""
    rng = np.random.default_rng(seed)
    rr, cc = np.mgrid[:det, :det].astype(np.float64)
    data = np.empty((*scan_shape, det, det), dtype=np.float32)
    for j in range(scan_shape[1]):
        e_cc = max_strain * j / (scan_shape[1] - 1)
        pattern = np.zeros((det, det))
        for h in range(-4, 5):
            for k in range(-4, 5):
                g = np.array([h * spacing, k * spacing / (1 + e_cc)])
                pattern += np.exp(-np.hypot(*g) / 25.0) * np.exp(
                    -((rr - det / 2 - g[0]) ** 2 + (cc - det / 2 - g[1]) ** 2) / (2 * 1.5**2)
                )
        for i in range(scan_shape[0]):
            data[i, j] = rng.poisson(pattern * 400) / 400.0
    return data


dataset = Dataset4dstem.from_array(synthetic_lattice_4dstem(), name="synthetic strain ramp")
dataset

## Pick the lattice and the peaks

Click two non-collinear lattice peaks on the mean transform: g1 first, then g2.
- Drag a vector tip to re-seed it. The lattice it predicts follows the pointer.
- Switch on **Multi-peak**, then drag **Threshold** and **Spacing**. The dashed candidate
  peaks preview what quantem will fit, and releasing a slider runs the refinement.
- **Mode**, **Gamma**, **Hann** and **Mask** change the transform itself.
- **Fit** runs every scan position on `device`.

In [ ]:
widget = ShowStrain(
    dataset,
    mode="gamma",
    gamma=0.5,
    refine_all_peaks=True,
    threshold_percentile=0.99,
    min_peak_spacing=4,
    refine_radius_px=1.5,
    device="auto",
)
widget

Vectors can also be set from code. They are `(row, col)` offsets from the transform centre, in transform pixels:

In [ ]:
widget.g1 = [8, 0]
widget.g2 = [0, 8]
widget.g1_refined, widget.g2_refined, widget.n_peaks

## Fit and continue in quantem

In [ ]:
strain = widget.fit()
print(widget.status_message)
print("mean e_cc per scan column (%):", np.round(strain.e_cc.array.mean(axis=0) * 100, 2))

Drag a box on a strain map to choose the reference region, for example the left (unstrained) columns. The same region from code, which recomputes the strain without refitting:

In [ ]:
widget.reference_region = [0, 24, 0, 3]  # [row0, row1, col0, col1], half-open scan indices
print(widget.reference_message)
print("mean e_cc per scan column (%):", np.round(widget.strain_map.e_cc.array.mean(axis=0) * 100, 2))

`strain` is a quantem `StrainMap`, and `widget.workflow` is the
`StrainMapAutocorrelation` in its current state. Use them for `plot_strain`,
`update_reference`, `estimate_strain_precision` and saving, exactly as in a script.
`widget.save("strain_settings.json")` writes the parameters, which
`load_state_dict` restores on another scan.